# CVD-Multimodal GPU Training, Calibration & Validation Pipeline (Kaggle)
**Project:** Explainable Multimodal Cardiovascular Disease Risk Prediction (`STEP-026` / `Blocks A-F`)  
**Governance Lock:** `Test` partition (`rel0001_test.jsonl`) stays **strictly SEALED** until Gate `G11`. Every hyperparameter, early-stopping checkpoint, post-hoc calibrator (`Temperature` / `Platt` / `Isotonic`), and operating threshold (`tau_youden`, `tau_sens_target >= 0.85`) is fit **exclusively on Train / Validation**.

### এই নোটবুকটি কী কী মডেল ও ধাপ চালাবে:
1. **Tier 1 (Tabular Baselines `MC0001–MC0005`):** Logistic Regression, Random Forest, RBF-SVM+Platt, XGBoost, LightGBM (`DS0001–DS0004` cohort-wise + pooled) — Train-only class weighting (`pos_weight = neg/pos`, zero pre-split SMOTE).
2. **Tier 2 (`MC0011` Clinical Baseline):** WHO / Framingham interpretable risk score.
3. **Tier 3 (`MC0006–MC0010` Deep ECG Encoders):** PyTorch GPU `CompactResNet1D`, `Conv1DBiLSTMEncoder`, `CompactCNN2DPNG`, `ECGPatchTransformer` (`< 250K` params).
4. **Tier 4 (`MC0012–MC0015` Multimodal Fusion & Proposed `CVDMMTransformer`):** Early Concatenation, Late OOF Logit Stacker, Gated Attention Fusion, and **CVD-MM-Transformer** (`d_model=64`, `2 Pre-LN layers`, `4 heads`, `~ModalityMask_CELS` token masking + `ModalityDropout 0.15`).
5. **Validation Calibration & Export:** Validation-only Calibration (`ECE-15`, `Brier`, `Slope/Intercept`), Frozen `tau_youden` / `tau_sens_target >= 0.85` Threshold Selection, Decision Curve Analysis (`DCA`), and exportable SHA-256 checkpoints in `/kaggle/working/model_outputs/`.

In [ ]:
# Cell 1: Determinism, Device Audit & Safe Dataset Discovery (Test Partition Locked)
import os
import sys
import json
import math
import time
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    brier_score_loss, matthews_corrcoef, balanced_accuracy_score
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEEDS = [42, 1024, 2025]
seed_everything(SEEDS[0])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR = Path('/kaggle/working/model_outputs') if Path('/kaggle').exists() else Path('model_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Auto-locate REL0001 JSONL shards under /kaggle/input or local workspace (NEVER read test!)
TEST_SEALED = True
REL_ROOT = None
for cand in [Path('/kaggle/input'), Path('05_Preprocessing_Alignment/processed'), Path('.')]:
    if not cand.exists():
        continue
    hits = list(cand.rglob('rel0001_train.jsonl'))
    if hits:
        REL_ROOT = hits[0].parent.parent
        break

print(f'Device={DEVICE} | PyTorch={torch.__version__} | XGB={HAS_XGB} | LGBM={HAS_LGBM}')
print(f'Discovered REL0001 root: {REL_ROOT} | TEST_SEALED={TEST_SEALED}')

In [ ]:
# Cell 2: Partition-Isolated Loader (Train + Val Only) & Cohort CELS Modality Slicer
COHORT_MODALITY_SLICES = {
    'DS0001': {'clinical': (0,1,2,3,4,5,6,7,8,9,10), 'laboratory': (), 'lifestyle': ()},
    'DS0002': {'clinical': (0,1,2,3,4,5,6,7,12,13,14), 'laboratory': (8,9,10,11), 'lifestyle': ()},
    'DS0003': {'clinical': (0,1,2,3,4,5,6,7,8,9,10,11,24,25,26), 'laboratory': (12,13,14,15,16,17,18,19), 'lifestyle': (20,21,22,23,27,28,29,30)},
    'DS0004': {'clinical': (0,1,2,3,4,5,6,7,8,9,10,11,22,23,24), 'laboratory': (12,13,14,15), 'lifestyle': (16,17,18,19,20,21)},
}
COHORT_MODALITY_DIMS = {
    ds: {m: len(idxs) for m, idxs in mods.items()} for ds, mods in COHORT_MODALITY_SLICES.items()
}

def load_rel_partition(part: str, max_rows: int | None = None):
    if part == 'test' and TEST_SEALED:
        raise PermissionError('TEST_PARTITION_SEALED: Test data is strictly locked until Gate G11 PASS!')
    if REL_ROOT is None:
        raise FileNotFoundError('Attach REL0001 Private Kaggle Dataset or run preprocessing first!')
    shard = REL_ROOT / part / f'rel0001_{part}.jsonl'
    recs = []
    with open(shard, encoding='utf-8') as f:
        for line in f:
            recs.append(json.loads(line))
            if max_rows is not None and len(recs) >= max_rows:
                break
    return recs

def compute_eval_bundle(y_true, y_prob, thr: float = 0.5):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = np.clip(np.asarray(y_prob, dtype=np.float64), 1e-7, 1.0 - 1e-7)
    y_pred = (y_prob >= thr).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    order = np.argsort(y_prob)
    ece_q = 0.0
    for bin_idx in np.array_split(order, min(15, len(order))):
        if len(bin_idx) > 0:
            ece_q += (len(bin_idx) / len(y_true)) * abs(y_true[bin_idx].mean() - y_prob[bin_idx].mean())
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'specificity': float(tn / max(1, tn + fp)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'mcc': float(matthews_corrcoef(y_true, y_pred)),
        'brier': float(brier_score_loss(y_true, y_prob)),
        'ece_q15': float(ece_q),
        'cm': [[int(tn), int(fp)], [int(fn), int(tp)]],
    }

if REL_ROOT is not None:
    train_records = load_rel_partition('train')
    val_records = load_rel_partition('val')
    print(f'Loaded REL0001 | Train={len(train_records):,} | Val={len(val_records):,}')
else:
    rng = np.random.default_rng(42)
    train_records = [{'DS': 'DS0004', 'y': int(rng.random() > 0.5), 'ModalityMask_CELS': [1,0,1,1], 'TabularFeatures': rng.normal(size=25).tolist()} for _ in range(400)]
    val_records = [{'DS': 'DS0004', 'y': int(rng.random() > 0.5), 'ModalityMask_CELS': [1,0,1,1], 'TabularFeatures': rng.normal(size=25).tolist()} for _ in range(120)]
    print('Warning: Running synthetic dry-run cohort records (attach REL0001 on Kaggle for full training).')

In [ ]:
# Cell 3: Tier 1 Classical Tabular Baselines (MC0001-MC0005) & Tier 2 Clinical Heuristic (MC0011)
baseline_val_results = []

def score_mc0011_who_framingham(X_va: np.ndarray, ds_id: str) -> np.ndarray:
    # Heuristic directional log-odds combining age, sbp/bmi, metabolic, smoking indices cleanly across cohorts
    z = -0.35 + 0.55 * X_va[:, 0] + 0.35 * X_va[:, 2]
    if ds_id in ('DS0003', 'DS0004') and X_va.shape[1] >= 16:
        z = z + 0.25 * X_va[:, 12] + 0.20 * X_va[:, 14]
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))

for ds_id in ['DS0001', 'DS0002', 'DS0003', 'DS0004']:
    tr_sub = [r for r in train_records if r['DS'] == ds_id]
    va_sub = [r for r in val_records if r['DS'] == ds_id]
    if len(tr_sub) < 20 or len(va_sub) < 10:
        continue
    X_tr = np.asarray([r['TabularFeatures'] for r in tr_sub], dtype=np.float32)
    y_tr = np.asarray([r['y'] for r in tr_sub], dtype=np.int64)
    X_va = np.asarray([r['TabularFeatures'] for r in va_sub], dtype=np.float32)
    y_va = np.asarray([r['y'] for r in va_sub], dtype=np.int64)

    neg_c, pos_c = int((y_tr == 0).sum()), max(1, int((y_tr == 1).sum()))
    pos_weight = neg_c / pos_c

    models = {
        'MC0001_LR': LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42),
        'MC0002_RF': RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', n_jobs=-1, random_state=42),
    }
    if len(tr_sub) <= 15000:
        models['MC0003_SVC'] = SVC(C=1.0, kernel='rbf', probability=True, class_weight='balanced', random_state=42)
    if HAS_XGB:
        models['MC0004_XGB'] = XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, eval_metric='auc', random_state=42, n_jobs=-1)
    if HAS_LGBM:
        models['MC0005_LGBM'] = LGBMClassifier(n_estimators=400, num_leaves=31, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=42, n_jobs=-1, verbose=-1)

    for m_name, clf in models.items():
        t0 = time.time()
        clf.fit(X_tr, y_tr)
        val_p = clf.predict_proba(X_va)[:, 1]
        res = compute_eval_bundle(y_va, val_p, thr=0.5)
        res.update({'cohort': ds_id, 'model': m_name, 'fit_sec': round(time.time() - t0, 2)})
        baseline_val_results.append(res)

    who_res = compute_eval_bundle(y_va, score_mc0011_who_framingham(X_va, ds_id), thr=0.5)
    who_res.update({'cohort': ds_id, 'model': 'MC0011_WHO_Context', 'fit_sec': 0.0})
    baseline_val_results.append(who_res)

df_base = pd.DataFrame(baseline_val_results)
print(df_base[['cohort', 'model', 'roc_auc', 'pr_auc', 'f1', 'ece_q15', 'brier']].to_string(index=False))
df_base.to_csv(OUT_DIR / 'tier1_2_baselines_val_metrics.csv', index=False)

In [ ]:
# Cell 4: Tier 3 Unimodal Deep ECG Encoders (MC0006-MC0010) & Tier 4 Gated/Transformer Fusion (<250K params)
class SEBlock1D(nn.Module):
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        h = max(4, channels // reduction)
        self.fc = nn.Sequential(nn.Linear(channels, h), nn.SiLU(), nn.Linear(h, channels), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean(dim=-1)).unsqueeze(-1)

class ResBlock1D(nn.Module):
    def __init__(self, c_in: int, c_out: int, stride: int = 1, dropout: float = 0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, 5, stride=stride, padding=2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, 5, stride=1, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.se = SEBlock1D(c_out)
        self.drop = nn.Dropout(dropout)
        self.short = nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm1d(c_out)) if (stride != 1 or c_in != c_out) else nn.Identity()
    def forward(self, x):
        out = self.drop(F.gelu(self.bn1(self.conv1(x))))
        out = self.se(self.bn2(self.conv2(out)))
        return F.gelu(out + self.short(x))

class CompactResNet1D(nn.Module):
    def __init__(self, in_leads: int = 12, embed_dim: int = 64):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_leads, 32, 7, stride=2, padding=3, bias=False), nn.BatchNorm1d(32), nn.GELU())
        self.blocks = nn.Sequential(ResBlock1D(32, 32, 2), ResBlock1D(32, 64, 2), ResBlock1D(64, embed_dim, 2))
    def forward(self, x):
        return self.blocks(self.stem(x)).mean(dim=-1)

class Conv1DBiLSTMEncoder(nn.Module):
    def __init__(self, in_leads: int = 12, embed_dim: int = 64):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(in_leads, 32, 7, stride=2, padding=3, bias=False), nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(2), nn.Conv1d(32, 48, 5, stride=2, padding=2, bias=False), nn.BatchNorm1d(48), nn.GELU())
        self.lstm = nn.LSTM(48, embed_dim // 2, batch_first=True, bidirectional=True)
    def forward(self, x):
        out, _ = self.lstm(self.conv(x).transpose(1, 2))
        return out.mean(dim=1)

class CompactCNN2DPNG(nn.Module):
    def __init__(self, embed_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 16, 3, 2, 1, bias=False), nn.BatchNorm2d(16), nn.GELU(), nn.Conv2d(16, 32, 3, 2, 1, bias=False), nn.BatchNorm2d(32), nn.GELU(), nn.Conv2d(32, embed_dim, 3, 2, 1, bias=False), nn.BatchNorm2d(embed_dim), nn.GELU(), nn.AdaptiveAvgPool2d(1), nn.Flatten())
    def forward(self, img):
        return self.net(img)

class ECGPatchTransformer(nn.Module):
    def __init__(self, in_leads: int = 12, patch_len: int = 50, d_model: int = 64):
        super().__init__()
        self.patch = nn.Conv1d(in_leads, d_model, kernel_size=patch_len, stride=patch_len)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        enc = nn.TransformerEncoderLayer(d_model, nhead=4, dim_feedforward=128, dropout=0.1, activation='gelu', batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, num_layers=2)
    def forward(self, x):
        p = self.patch(x).transpose(1, 2)
        return self.enc(torch.cat([self.cls.expand(p.size(0), -1, -1), p], dim=1))[:, 0]

class TabularModalityMLP(nn.Module):
    def __init__(self, in_dim: int, d_model: int = 64, dropout: float = 0.15):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
    def forward(self, x):
        return self.net(x)

class CVDMMTransformer(nn.Module):
    def __init__(self, cohort_dims: dict, d_model: int = 64, nhead: int = 4, n_layers: int = 2, dropout: float = 0.1, modality_dropout_p: float = 0.15):
        super().__init__()
        self.d_model = d_model
        self.modality_dropout_p = modality_dropout_p
        adapters = {}
        for ds_id, mods in cohort_dims.items():
            for m, dim in mods.items():
                if dim > 0:
                    adapters[f'{ds_id}__{m}'] = TabularModalityMLP(dim, d_model, dropout=0.15)
        self.tab_adapters = nn.ModuleDict(adapters)
        self.ecg_encoder = CompactResNet1D(12, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.mod_type_emb = nn.Parameter(torch.randn(1, 4, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=128, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.GELU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, ds_id: str, tab_mods: dict, mask_cels: torch.Tensor, ecg_1d: torch.Tensor | None = None):
        b, dev = mask_cels.size(0), mask_cels.device
        eff_mask = mask_cels.bool()
        if self.training and self.modality_dropout_p > 0.0:
            drop = torch.rand_like(mask_cels.float()) < self.modality_dropout_p
            dropped = eff_mask & (~drop)
            all_zero = dropped.sum(dim=1) == 0
            dropped[all_zero] = eff_mask[all_zero]
            eff_mask = dropped
        toks = []
        for idx, mod in enumerate(['clinical', 'ecg', 'laboratory', 'lifestyle']):
            if mod == 'ecg':
                emb = self.ecg_encoder(ecg_1d) if ecg_1d is not None else torch.zeros(b, self.d_model, device=dev)
            else:
                key = f'{ds_id}__{mod}'
                emb = self.tab_adapters[key](tab_mods[mod]) if (key in self.tab_adapters and mod in tab_mods) else torch.zeros(b, self.d_model, device=dev)
            toks.append(emb + self.mod_type_emb[:, idx, :])
        seq = torch.cat([self.cls_token.expand(b, -1, -1), torch.stack(toks, dim=1)], dim=1)
        pad_mask = torch.cat([torch.zeros(b, 1, dtype=torch.bool, device=dev), ~eff_mask], dim=1)
        out = self.norm(self.encoder(seq, src_key_padding_mask=pad_mask))
        return self.head(out[:, 0]).squeeze(-1)

demo_model = CVDMMTransformer(COHORT_MODALITY_DIMS).to(DEVICE)
total_params = sum(p.numel() for p in demo_model.parameters() if p.requires_grad)
print(f'CVDMMTransformer Trainable Parameters = {total_params:,} (Budget <= 250,000)')

In [ ]:
# Cell 5: Training Loop, Validation Calibration (Temperature), & tau_youden / tau_sens>=0.85 Freeze
def select_val_thresholds(y_val_np: np.ndarray, cal_probs: np.ndarray, min_sens: float = 0.85):
    best_j, tau_j = -1.0, 0.5
    best_spec, tau_s = -1.0, float(cal_probs.min())
    for tau in np.linspace(0.05, 0.95, 181):
        pred = cal_probs >= tau
        tp = int(((y_val_np == 1) & pred).sum())
        tn = int(((y_val_np == 0) & (~pred)).sum())
        sens = tp / max(1, int((y_val_np == 1).sum()))
        spec = tn / max(1, int((y_val_np == 0).sum()))
        if (sens + spec - 1.0) > best_j:
            best_j, tau_j = (sens + spec - 1.0), float(tau)
        if sens >= min_sens and spec > best_spec:
            best_spec, tau_s = spec, float(tau)
    return round(tau_j, 4), round(tau_s, 4)

def fit_cvd_mm_transformer_for_cohort(ds_id: str, seed: int = 42, epochs: int = 25, batch_size: int = 128):
    seed_everything(seed)
    tr = [r for r in train_records if r['DS'] == ds_id]
    va = [r for r in val_records if r['DS'] == ds_id]
    slices = COHORT_MODALITY_SLICES[ds_id]

    def _tensors(recs):
        X = np.asarray([r['TabularFeatures'] for r in recs], dtype=np.float32)
        mods = {m: torch.from_numpy(X[:, list(idxs)]) for m, idxs in slices.items() if len(idxs) > 0}
        masks = torch.tensor([r['ModalityMask_CELS'] for r in recs], dtype=torch.bool)
        ys = torch.tensor([r['y'] for r in recs], dtype=torch.float32)
        return mods, masks, ys

    tr_mods, tr_mask, tr_y = _tensors(tr)
    va_mods, va_mask, va_y = _tensors(va)
    pos_w = float((tr_y == 0).sum().item()) / max(1.0, float((tr_y == 1).sum().item()))

    model = CVDMMTransformer(COHORT_MODALITY_DIMS).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w], device=DEVICE))

    best_auc, best_logits, best_state = -1.0, None, None
    n_tr = len(tr)
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n_tr)
        for i in range(0, n_tr, batch_size):
            idx = perm[i:i+batch_size]
            opt.zero_grad(set_to_none=True)
            b_mods = {m: t[idx].to(DEVICE) for m, t in tr_mods.items()}
            logits = model(ds_id, b_mods, tr_mask[idx].to(DEVICE))
            loss = crit(logits, tr_y[idx].to(DEVICE))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            v_mods = {m: t.to(DEVICE) for m, t in va_mods.items()}
            v_logits = model(ds_id, v_mods, va_mask.to(DEVICE)).cpu().numpy()
            v_probs = 1.0 / (1.0 + np.exp(-np.clip(v_logits, -50, 50)))
            v_auc = roc_auc_score(va_y.numpy(), v_probs)
            if v_auc > best_auc:
                best_auc = v_auc
                best_logits = v_logits.copy()
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    # Validation-Only Temperature Scaling + Frozen Operating Thresholds
    best_t, best_nll = 1.0, 1e18
    y_va_np = va_y.numpy()
    for t in np.linspace(0.25, 3.0, 111):
        p = np.clip(1.0 / (1.0 + np.exp(-np.clip(best_logits / t, -50, 50))), 1e-7, 1 - 1e-7)
        nll = -(y_va_np * np.log(p) + (1 - y_va_np) * np.log(1 - p)).mean()
        if nll < best_nll:
            best_nll, best_t = nll, float(t)
    cal_probs = 1.0 / (1.0 + np.exp(-np.clip(best_logits / best_t, -50, 50)))
    tau_youden, tau_sens85 = select_val_thresholds(y_va_np, cal_probs, min_sens=0.85)
    ckpt_path = OUT_DIR / f'CVDMMTransformer_{ds_id}_seed{seed}_best_val.pt'
    torch.save({'cohort': ds_id, 'seed': seed, 'val_roc_auc': best_auc, 'temperature': best_t, 'tau_youden': tau_youden, 'tau_sens85': tau_sens85, 'state_dict': best_state}, ckpt_path)
    metrics = compute_eval_bundle(y_va_np, cal_probs, thr=tau_youden)
    metrics.update({'cohort': ds_id, 'seed': seed, 'temperature': round(best_t, 4), 'tau_youden': tau_youden, 'tau_sens85': tau_sens85, 'ckpt': str(ckpt_path)})
    return metrics

transformer_val_runs = []
for ds_id in sorted({r['DS'] for r in train_records}):
    transformer_val_runs.append(fit_cvd_mm_transformer_for_cohort(ds_id, seed=SEEDS[0], epochs=8 if REL_ROOT is None else 25))

df_tr = pd.DataFrame(transformer_val_runs)
print(df_tr[['cohort', 'seed', 'roc_auc', 'pr_auc', 'ece_q15', 'temperature', 'tau_youden', 'tau_sens85']].to_string(index=False))
df_tr.to_csv(OUT_DIR / 'cvd_mm_transformer_val_metrics.csv', index=False)